# 自动发送带附件的邮件

## 1. 课程目标
本实训将教你如何使用Python编写程序，自动发送包含Excel、Word和PDF附件的邮件。

## 2. 发送邮件的 Python 库简介
- smtplib (Python 内置，用于发送邮件)
- email (Python 内置，用于构建邮件内容)
- openpyxl (用于生成 Excel 文件)
- python-docx (用于生成 Word 文件)
- reportlab (用于生成 PDF 文件)

## 3. 概念解释
邮件自动化发送的核心是理解其背后的通信协议。其中，**SMTP (Simple Mail Transfer Protocol)**，即简单邮件传输协议，是用于发送邮件的标准协议。你可以把它想象成邮局的投递员，负责将你写好的信件从你的邮箱服务器发送到接收方的邮箱服务器。

SMTP 协议定义了邮件客户端（如你的 Python 脚本）如何与邮件服务器通信，以及邮件服务器之间如何传递邮件。

**SMTP 的主要特点：**

*   **文本传输**：SMTP 协议最初设计为传输纯文本信息。
*   **客户端/服务器模式**：你的邮件客户端（Python 脚本）充当客户端，与邮件服务器（如 QQ 邮箱的 SMTP 服务器）进行通信。
*   **端口**：SMTP 通常使用 25 端口（非加密），或者 465 端口（SSL/TLS 加密），以及 587 端口（STARTTLS 加密）。为了安全起见，我们通常会使用 465 或 587 端口。
*   **认证**：为了防止垃圾邮件和非法使用，大多数 SMTP 服务器都要求进行身份认证，通常是使用用户名和密码（或授权码）。

```mermaid
flowchart LR
    A[Python客户端] -->|连接| B[SMTP服务器]
    A -->|登录认证| B
    A -->|发送邮件| B
    B -->|转发邮件| C[收件人服务器]
    C -->|存储邮件| D[收件邮箱]
    E[邮件客户端] -->|收取邮件| D
    
    style A fill:#e1f5fe
    style E fill:#e1f5fe
    style B fill:#fff3e0
    style C fill:#fff3e0
    style D fill:#f1f8e9
```

**图表说明:**

*   **发件人客户端 (Python)**：你的 Python 脚本作为客户端，负责发起邮件发送请求。
*   **发件人 SMTP 服务器**：负责接收发件人客户端的邮件，并将其转发到收件人 SMTP 服务器。
*   **身份认证 (AUTH LOGIN)**：在发送邮件前，客户端需要向服务器提供凭证（授权码）进行身份验证。
*   **发送邮件指令**：客户端通过一系列 SMTP 命令（如 `MAIL FROM` 指定发件人，`RCPT TO` 指定收件人，`DATA` 发送邮件内容）与服务器交互。
*   **收件人 SMTP 服务器**：接收来自发件人 SMTP 服务器的邮件，并将其投递到收件人的邮箱。
*   **收件人邮箱**：邮件最终存储的地方。
*   **收件人客户端**：收件人通过邮件客户端（如 Outlook, Gmail 或其他邮件应用）使用 IMAP 或 POP3 协议从邮箱服务器收取邮件。

## 4. 实训过程

### 步骤 1: 导入所需库

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
import openpyxl
from docx import Document
from reportlab.pdfgen import canvas
import io
import os

### 步骤 2: 编写创建文件的方法
**创建 excel 文件**

In [ ]:
def create_excel_file():
    """创建Excel文件"""
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "学生成绩单"
    
    # 添加数据
    data = [
        ["学号", "姓名", "成绩"],
        ["001", "张三", 95],
        ["002", "李四", 88],
        ["003", "王五", 92]
    ]
    
    for row in data:
        ws.append(row)
    
    # 保存文件
    filename = "学生成绩单.xlsx"
    wb.save(filename)
    return filename

**创建 word 文件**

In [ ]:
def create_word_file():
    """创建Word文件"""
    doc = Document()
    
    # 添加标题
    doc.add_heading('实训报告', 0)
    
    # 添加段落
    doc.add_paragraph('本次实训内容：自动发送带附件的邮件')
    doc.add_paragraph('完成情况：已成功完成所有任务')
    
    # 保存文件
    filename = "实训报告.docx"
    doc.save(filename)
    return filename

**创建 pdf 文件**

In [ ]:
def create_pdf_file():
    """创建PDF文件"""
    filename = "说明文档.pdf"
    
    # 创建PDF
    c = canvas.Canvas(filename)
    
    # 添加文本
    c.drawString(100, 750, "Python邮件自动发送系统")
    c.drawString(100, 730, "功能说明：")
    c.drawString(100, 710, "1. 自动生成Excel、Word、PDF文件")
    c.drawString(100, 690, "2. 自动发送带附件的邮件")
    c.drawString(100, 670, "3. 支持QQ邮箱发送")
    
    # 保存文件
    c.save()
    return filename

### 步骤 3：编写发送邮件的方法

In [ ]:
def send_email(sender_email, sender_auth_code, receiver_email, attachments):
    """发送带附件的邮件"""
    # 设置邮件内容
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = "Python自动发送的测试邮件"
    
    # 邮件正文
    body = """
    <h2>Python自动发送邮件测试</h2>
    <p>这是一封由Python程序自动发送的测试邮件。</p>
    <p>邮件包含三个附件：Excel文件、Word文件和PDF文件。</p>
    <p>请查收！</p>
    """
    msg.attach(MIMEText(body, 'html'))
    
    # 添加附件
    for file_path in attachments:
        with open(file_path, "rb") as f:
            part = MIMEApplication(f.read(), Name=os.path.basename(file_path))
        part['Content-Disposition'] = f'attachment; filename="{os.path.basename(file_path)}"'
        msg.attach(part)
    
    # 发送邮件
    try:
        # QQ邮箱SMTP服务器设置
        server = smtplib.SMTP_SSL('smtp.qq.com', 465)
        server.login(sender_email, sender_auth_code)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        server.quit()
        print("邮件发送成功！")
    except Exception as e:
        print(f"邮件发送失败: {e}")

## 5. 实际应用场景


首先我们需要获取我们 qq 邮箱的授权码。


<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/k9/middle7/20250904152625044.png" width="1200px"/></div>
</div>

保存好你的内存码。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/k9/middle7/20250904152625045.png" width="400px"/></div>
</div>



In [ ]:
print("=== Python自动发送带附件邮件系统 ===")

# 获取用户输入
sender_email = input("请输入你的QQ邮箱: ")
sender_auth_code = input("请输入QQ邮箱授权码: ")
receiver_email = input("请输入收件人邮箱: ")

print("正在生成附件文件...")

# 创建附件文件
excel_file = create_excel_file()
word_file = create_word_file()
pdf_file = create_pdf_file()

attachments = [excel_file, word_file, pdf_file]

print("文件生成完成！")
print("正在发送邮件...")

# 发送邮件
send_email(sender_email, sender_auth_code, receiver_email, attachments)

# 清理生成的文件（可选）
clean_files = input("是否删除生成的附件文件？(y/n): ")
if clean_files.lower() == 'y':
    for file in attachments:
        os.remove(file)
    print("文件已清理")

print("程序执行完毕！")


<div class='insertContainerBox row'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/k9/middle7/20250905100812648.png" width="800px"/></div>
</div>
